# BLISS AstroPy Affiliate Package Tutorial

## Introduction

Bayesian Light Source Separator (BLISS) is a Bayesian method for deblending and cataloging light sources.

## Installation

In [ ]:
%env BLISS_HOME=/home/zhteoh/730-astropy-integration

In [ ]:
!pip install -e $BLISS_HOME

# Tutorial

## Train the model

### Generate synthetic image data

In [ ]:
from bliss.api import generate

In [ ]:
generate(
    n_batches=3, 
    batch_size=64, 
    max_images_per_file=128, 
    cached_data_path="/data/scratch/zhteoh/730-tutorial/dataset"
)

#### Pass additional custom configuration parameters

In [ ]:
generate(
    n_batches=3,  # required
    batch_size=64,  # required
    max_images_per_file=128,  # required
    cached_data_path="/data/scratch/zhteoh/730-tutorial/dataset",  # required
    simulator={"prior": {"mean_sources": 0.02}},  # optional
    generate={"file_prefix": "dataset"},  # optional
)

In [ ]:
# Check that the dataset is generated
!ls /data/scratch/zhteoh/730-tutorial/dataset
!du -sh /data/scratch/zhteoh/730-tutorial/dataset
# !cat /data/scratch/zhteoh/730-tutorial/dataset/hparams.yaml

import torch
with open("/data/scratch/zhteoh/730-tutorial/dataset/dataset_0.pt", "rb") as f:
    dataset = torch.load(f)
print(len(dataset))
print(dataset[0]["images"].shape)

### Train the model

In [ ]:
from bliss.api import train

#### Without pretrained weights

In [ ]:
train(
    weight_save_path="/data/scratch/zhteoh/730-tutorial/output/tutorial_encoder/0.pt",
)

#### With pretrained weights

Download our relevant pretrained weights for your sky survey.

In [ ]:
from bliss.api import load_pretrained_weights_for_survey

import os
assert os.path.exists("/data/scratch/zhteoh/730-tutorial/pretrained_weights")

load_pretrained_weights_for_survey(
    survey="sdss",
    pretrained_weights_path="/data/scratch/zhteoh/730-tutorial/pretrained_weights/sdss_pretrained.pt",
)

#### Train on cached generated disk dataset

In [ ]:
from bliss.api import train_on_cached_data

In [ ]:
train_on_cached_data(
    weight_save_path="/data/scratch/zhteoh/730-tutorial/output/tutorial_encoder/0.pt",
    cached_data_path="/data/scratch/zhteoh/730-tutorial/dataset",
    train_n_batches=2,
    batch_size=64,
    val_split_file_idxs=[1],
    training={"pretrained_weights": "/data/scratch/zhteoh/730-tutorial/pretrained_weights/sdss_pretrained.pt"}
)

## Run the model

### Using sample dataset

#### Download the sample dataset

In [ ]:
from bliss.surveys.sdss import PhotoFullCatalog, SloanDigitalSkySurvey

sdss_data_path = "/home/zhteoh/730-astropy-integration/data/sdss"
photo_cat = PhotoFullCatalog.from_file(sdss_data_path, run=94, camcol=1, field=12, band=2)
sdss = SloanDigitalSkySurvey(sdss_data_path, 94, 1, (12,), (2,))

#### Get predictions for the sample dataset

In [ ]:
sdss_cfg = {
    "predict": {
        "_target_": "bliss.surveys.sdss.SloanDigitalSkySurvey",
        "sdss_dir": "${paths.sdss}",
        "run": 94,
        "camcol": 1,
        "fields": [12],
        "bands": [2],
    },
    "weight_save_path": "/home/zhteoh/727-optimize-encoder/case_studies/optimize_encoder_727/output/encoder/sdss-finetuned-0.006",
}

from bliss.conf.igs import base_config
base_cfg = base_config()
from omegaconf import OmegaConf
conf = OmegaConf.merge(base_cfg, sdss_cfg)


In [ ]:
from bliss.predict import predict, predict_sdss

est_cat, crop_img, crop_bg, _ = predict_sdss(conf)

# TODO: export catalog as fits file

In [ ]:
from IPython.display import display
from IPython.core.display import HTML

with open("./predict.html", "r") as f:
    html_str = f.read()
    display(HTML(html_str))

#### Evaluate prediction

In [ ]:
import torch

from bliss.metrics import BlissMetrics

est_cat_cuda = est_cat.to(torch.device("cpu"))
photo_cat_cuda = photo_cat.to(torch.device("cpu"))

metrics = BlissMetrics()
results = metrics(est_cat_cuda, photo_cat_cuda)

print(results)

### Using user-specified dataset

#### Download online dataset

In [ ]:
from bliss.api import load_survey

sp, im = load_survey("sdss")

